<!-- CODEx bilingual cell explanation: start -->
### Cell 01 — 环境、模型库与验证参数 / Environment, model libraries, and validation settings

**中文说明**：导入 scikit-learn、XGBoost、LightGBM、joblib 和绘图库，建立 Step 3 输出目录，设置目标变量、随机种子、内层交叉验证折数和快速验证开关。所有预处理均在 Pipeline 内完成，避免训练/测试信息泄漏。

**输入与依赖**：依赖本地 Python/Jupyter 环境、项目根目录和后续单元需要共享的配置参数。

**主要输出**：建立路径、随机种子、绘图样式、配置字典或输出目录等基础对象。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Imports scikit-learn, XGBoost, LightGBM, joblib, and plotting libraries; creates Step 3 output directories; and defines the target, random seed, inner cross-validation folds, and fast-validation switch. All preprocessing is placed inside Pipelines to avoid train/test information leakage.

**Inputs and dependencies**: Depends on the local Python/Jupyter environment, the project root, and shared configuration values used by later cells.

**Main outputs**: Creates paths, random seeds, plotting defaults, configuration dictionaries, or output directories.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:

import warnings
import os
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
})

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "input" / "Beijing.epw").exists() or (candidate / "data" / "step1_simulation_dataset.csv").exists():
            return candidate
        if (candidate / "AGENTS.md").exists() and (candidate / "outputs_step1").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "experiment_code"
DATA_DIR = PROJECT_ROOT / "data"
STEP2_DIR = PROJECT_ROOT / "outputs_step2"
OUT_DIR = PROJECT_ROOT / "outputs_step3"
FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "models"

PAPER_ASSETS_DIR = PROJECT_ROOT / "paper_assets"
PAPER_ASSETS_FIG_DIR = PAPER_ASSETS_DIR / "figures"
for search_path in [PROJECT_ROOT, CODE_DIR]:
    if search_path.exists() and str(search_path) not in sys.path:
        sys.path.insert(0, str(search_path))

for p in [OUT_DIR, FIG_DIR, MODEL_DIR, PAPER_ASSETS_DIR, PAPER_ASSETS_FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def save_figure(figure, path, **kwargs):
    """Save figures through a temporary file to avoid interrupted overwrites on Windows/Jupyter."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.stem}.__tmp__{path.suffix}")
    figure.savefig(tmp_path, **kwargs)
    tmp_path.replace(path)

DATASET_PATH = DATA_DIR / "step1_simulation_dataset.csv"
SRC_PATH = STEP2_DIR / "src_indices_bootstrap.csv"

TARGET = "eui_kwh_m2"
RANDOM_SEED = 42
FAST_MODE = os.environ.get("EUI_FAST_MODE", "0") == "1"
N_JOBS = 1 if FAST_MODE else -1
INNER_CV = 3 if FAST_MODE else 10
SEARCH_N_ITER = 3 if FAST_MODE else 20

<!-- CODEx bilingual cell explanation: start -->
### Cell 02 — 数据加载、Top-18 特征和训练测试划分 / Data loading, Top-18 features, and train/test split

**中文说明**：读取 Step 1 数据和 Step 2 SRC 排序，复用相同特征工程，选取 SRC 前 18 个变量，固定 80/20 随机训练测试划分，并导出变量范围和非核心变量中位数表。该 cell 是所有模型公平比较的数据基础。

**输入与依赖**：读取上游步骤生成的 CSV、模型清单或配置对象，并检查必要字段是否存在。

**主要输出**：生成清洗后的 DataFrame、特征列表、训练测试划分或供后续单元复用的中间变量。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Loads Step 1 data and Step 2 SRC ranking, applies the same feature engineering, selects the top 18 SRC variables, fixes an 80/20 random train/test split, and exports feature ranges plus non-core-feature medians. This cell is the common data basis for fair model comparison.

**Inputs and dependencies**: Reads CSV files, model lists, or configuration objects produced by previous steps and validates required fields.

**Main outputs**: Creates cleaned DataFrames, feature lists, train/test splits, or intermediate objects reused downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
assert DATASET_PATH.exists(), "Please complete Step 1 first"
assert SRC_PATH.exists(), "Please complete Step 2 first"

df = pd.read_csv(DATASET_PATH)
src_df = pd.read_csv(SRC_PATH)

# Use the same feature engineering as in Step 2.
df["orientation_sin"] = np.sin(np.deg2rad(df["orientation_deg"]))
df["orientation_cos"] = np.cos(np.deg2rad(df["orientation_deg"]))
df = pd.get_dummies(df, columns=["window_type_id"], prefix="window_type", drop_first=True)

analysis_features = [
    'insul_thick', 'wwr', 'wall_thick',
    'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
    'u_wall', 'u_roof', 'u_ground',
    'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
    'roof_insul_thick',
    'floor_num', 'footprint_area_m2', 'aspect_ratio', 'floor_height',
    'orientation_sin', 'orientation_cos',
    'public_area', 'room_area', 'room_count',
    'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
    'cool_set', 'heat_set', 'dhw_temp',
    'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
    'fresh_air_ach', 'operation_hours',
    'window_type_2', 'window_type_3'
]

src_df = src_df[src_df["feature"].isin(analysis_features)].copy()
top18 = src_df.sort_values("abs_SRC", ascending=False).head(18)["feature"].tolist()
fixed_features = [f for f in analysis_features if f not in top18]

X = df[top18].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    shuffle=True
)

top18_range_df = pd.DataFrame({
    "feature": top18,
    "min": [df[f].min() for f in top18],
    "max": [df[f].max() for f in top18],
    "mean": [df[f].mean() for f in top18],
    "median": [df[f].median() for f in top18],
})

fixed_constants_df = pd.DataFrame({
    "feature": fixed_features,
    "fixed_value": [df[f].median() for f in fixed_features]
})

pd.Series(top18, name="top18_variable_features").to_csv(
    OUT_DIR / "top18_variable_features.csv", index=False, encoding="utf-8-sig"
)
top18_range_df.to_csv(
    OUT_DIR / "top18_variable_ranges.csv", index=False, encoding="utf-8-sig"
)
fixed_constants_df.to_csv(
    OUT_DIR / "fixed_background_constants.csv", index=False, encoding="utf-8-sig"
)
print("Top18 features:", top18)
print("X shape:", X.shape)
print("Train/Test:", X_train.shape, X_test.shape)

<!-- CODEx bilingual cell explanation: start -->
### Cell 03 — 样本量快速核对 / Sample-count sanity check

**中文说明**：打印当前仿真数据集样本量，用于提醒作者当前是否仍处于 116 个样本的调试数据状态，还是已经完成 4,640 个可行样本的完整仿真。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Prints the current simulation-dataset sample count to remind the author whether the notebook is still using the 116-sample debug dataset or the completed 4,640-feasible-sample simulation dataset.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
print(len(df))

<!-- CODEx bilingual cell explanation: start -->
### Cell 04 — EUI 标签分布与统计 / EUI-label distribution and statistics

**中文说明**：绘制机器学习标签 EUI 的分布、均值和中位数，并导出摘要统计 CSV。该图用于判断模型训练目标是否存在异常偏态或离群值。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots the machine-learning target EUI distribution with mean and median markers and exports summary statistics. The figure checks whether the training label has unusual skewness or outliers.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- EUI-label distribution and summary-statistics export ----------
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Extract the EUI label from the training dataset.
eui_data = df[TARGET].dropna()

# Calculate summary statistics.
mean_val = eui_data.mean()
median_val = eui_data.median()
std_val = eui_data.std()

# Plot the distribution.
plt.figure(figsize=(8, 5.5), dpi=300)

sns.histplot(
    eui_data,
    bins=30,
    kde=True,
    stat="count"   # Use frequency/count on the y-axis instead of density.
)

plt.axvline(
    mean_val,
    color="#D62728",
    linestyle="--",
    linewidth=1.8,
    label=f"Mean = {mean_val:.1f}"
)

plt.axvline(
    median_val,
    color="#2CA02C",
    linestyle="-.",
    linewidth=1.8,
    label=f"Median = {median_val:.1f}"
)

plt.xlabel("EUI (kWh/(m2.a))", fontsize=12)
plt.ylabel("Frequency", fontsize=12)   # Label the y-axis as Frequency.
plt.title("EUI-label distribution for model-training samples", fontsize=13)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.legend(frameon=False, fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()

# Save the figure.
save_figure(plt.gcf(), FIG_DIR / "Fig_EUI_distribution_for_training_samples.png", dpi=300, bbox_inches="tight")
plt.show()

# Build the summary-statistics table.
summary_df = pd.DataFrame({
    "metric": ["mean", "median", "std"],
    "value": [mean_val, median_val, std_val]
})

# Export the CSV file.
summary_df.to_csv(FIG_DIR / "reconstructed_eui_summary_statistics.csv", index=False, encoding="utf-8-sig")

# Display the result.
print("EUI statistics for model training samples:")
print(summary_df)
print(f"\nCSV saved to: {FIG_DIR / 'reconstructed_eui_summary_statistics.csv'}")

<!-- CODEx bilingual cell explanation: start -->
### Cell 05 — 候选模型与超参数搜索空间 / Candidate models and hyperparameter search spaces

**中文说明**：定义线性、正则化、多项式、KNN、SVR、树模型、集成模型、XGBoost、LightGBM 和 MLP 的 Pipeline 与搜索空间。线性模型使用带标准化的 Pipeline，树模型使用中位数插补 Pipeline；搜索均在训练集内进行。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Defines Pipelines and search spaces for linear, regularised, polynomial, KNN, SVR, tree, ensemble, XGBoost, LightGBM, and MLP regressors. Linear models use imputation and scaling, tree models use imputation only, and all searches are performed within the training set.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import ExtraTreesRegressor

prep_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

prep_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

alphas = np.logspace(-4, 4, 40)

# ---------- 1) Models that do not require an external hyperparameter search ----------
models = {
    "Linear": Pipeline([
        ("prep", prep_linear),
        ("model", LinearRegression())
    ]),

    "RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("model", RidgeCV(alphas=alphas, cv=INNER_CV))
    ]),

    "LassoCV": Pipeline([
        ("prep", prep_linear),
        ("model", LassoCV(
            alphas=alphas,
            cv=INNER_CV,
            max_iter=50000,
            random_state=RANDOM_SEED
        ))
    ]),

    "ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=alphas,
            cv=INNER_CV,
            max_iter=50000,
            random_state=RANDOM_SEED
        ))
    ]),

    "Poly2-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=INNER_CV))
    ]),

    "Poly2-ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-3, 2, 20),
            cv=INNER_CV,
            max_iter=50000,
            random_state=RANDOM_SEED
        ))
    ]),

    "Poly2-Interaction-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=INNER_CV))
    ]),

    "Poly3-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-1, 5, 30), cv=INNER_CV))
    ]),
}

# ---------- 2) Models tuned with automatic search ----------
searchers = {
    "KNN": GridSearchCV(
        Pipeline([
            ("prep", prep_linear),
            ("model", KNeighborsRegressor())
        ]),
        param_grid={
            "model__n_neighbors": [3, 5, 7, 9, 11, 15],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        },
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        n_jobs=N_JOBS
    ),

    "SVR-RBF": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_linear),
            ("model", SVR(kernel="rbf"))
        ]),
        param_distributions={
            "model__C": np.logspace(-1, 2, 20),
            "model__epsilon": np.linspace(0.005, 0.2, 20),
            "model__gamma": ["scale", "auto"]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),

    "RandomForest": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=N_JOBS))
        ]),
        param_distributions={
            "model__n_estimators": [300, 500, 800, 1200],
            "model__max_depth": [None, 6, 8, 12, 16],
            "model__min_samples_split": [2, 4, 8],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),

    "ExtraTrees": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=N_JOBS))
        ]),
        param_distributions={
            "model__n_estimators": [300, 500, 800, 1200],
            "model__max_depth": [None, 6, 8, 12, 16],
            "model__min_samples_split": [2, 4, 8],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),

    "GB": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", GradientBoostingRegressor(random_state=RANDOM_SEED))
        ]),
        param_distributions={
            "model__n_estimators": [100, 200, 300, 500],
            "model__learning_rate": np.linspace(0.01, 0.15, 15),
            "model__max_depth": [2, 3, 4, 5],
            "model__subsample": [0.7, 0.8, 0.9, 1.0]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),

    "MLP": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_linear),
            ("model", MLPRegressor(
                random_state=RANDOM_SEED,
                max_iter=5000,
                early_stopping=True
            ))
        ]),
        param_distributions={
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "model__alpha": np.logspace(-5, -1, 10),
            "model__activation": ["relu", "tanh"]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),

    "DecisionTree": RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", DecisionTreeRegressor(random_state=RANDOM_SEED))
        ]),
        param_distributions={
            "model__max_depth": [None, 4, 6, 8, 12, 16],
            "model__min_samples_split": [2, 4, 8, 12],
            "model__min_samples_leaf": [1, 2, 4, 8]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    ),
}

# ---------- 3) Merge model sets ----------
all_estimators = {}
all_estimators.update(models)
all_estimators.update(searchers)

# ---------- 4) Add XGBoost / LightGBM if available ----------
if HAS_XGB:
    all_estimators["XGBoost"] = RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", XGBRegressor(
                objective="reg:squarederror",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS
            ))
        ]),
        param_distributions={
            "model__n_estimators": [200, 400, 600, 800],
            "model__max_depth": [3, 4, 5, 6],
            "model__learning_rate": np.linspace(0.02, 0.15, 10),
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    )

if HAS_LGBM:
    all_estimators["LightGBM"] = RandomizedSearchCV(
        Pipeline([
            ("prep", prep_tree),
            ("model", LGBMRegressor(random_state=RANDOM_SEED, verbose=-1))
        ]),
        param_distributions={
            "model__n_estimators": [200, 400, 600, 800],
            "model__learning_rate": np.linspace(0.02, 0.15, 10),
            "model__num_leaves": [15, 31, 63],
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
        },
        n_iter=SEARCH_N_ITER,
        cv=INNER_CV,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    )

search_spaces = {name: getattr(searcher, 'param_distributions', getattr(searcher, 'param_grid', None)) for name, searcher in searchers.items()}
if HAS_XGB:
    search_spaces['XGBoost'] = all_estimators['XGBoost'].param_distributions
if HAS_LGBM:
    search_spaces['LightGBM'] = all_estimators['LightGBM'].param_distributions

print("Total models:", len(all_estimators))
print(list(all_estimators.keys()))

<!-- CODEx bilingual cell explanation: start -->
### Cell 06 — 模型训练、交叉验证与测试集评估 / Model fitting, cross-validation, and test-set evaluation

**中文说明**：逐模型拟合训练集，提取最优估计器，在独立测试集上计算 R2、RMSE、MAE、MAPE，并在训练集内做交叉验证估计均值和方差。结果保存为 model_metrics.csv。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Fits each model on the training set, extracts the best estimator, evaluates R2, RMSE, MAE, and MAPE on the independent test set, and computes cross-validation means and variances within the training set. Results are saved as model_metrics.csv.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:

def fit_and_compare_models(X_train, y_train, X_test, y_test, estimators):
    rows = []
    fitted_models = {}
    search_objects = {}
    best_params_by_model = {}

    for name, est in estimators.items():
        est.fit(X_train, y_train)

        if hasattr(est, "best_estimator_"):
            best_model = est.best_estimator_
            best_params = est.best_params_
            cv_score = -est.best_score_
        else:
            best_model = est
            best_params = None
            cv_score = np.nan

        pred_train = best_model.predict(X_train)
        pred_test = best_model.predict(X_test)

        cv_result = cross_validate(
            best_model,
            X_train,
            y_train,
            cv=INNER_CV,
            scoring={
                "r2": "r2",
                "neg_rmse": "neg_root_mean_squared_error",
                "neg_mae": "neg_mean_absolute_error",
            },
            n_jobs=N_JOBS,
        )

        cv_r2_scores = cv_result["test_r2"]
        cv_rmse_scores = -cv_result["test_neg_rmse"]
        cv_mae_scores = -cv_result["test_neg_mae"]

        rows.append({
            "model": name,
            "train_r2": r2_score(y_train, pred_train),
            "test_r2": r2_score(y_test, pred_test),
            "test_rmse": np.sqrt(mean_squared_error(y_test, pred_test)),
            "test_mae": mean_absolute_error(y_test, pred_test),
            "test_mape": mean_absolute_percentage_error(y_test, pred_test),
            "cv_best_rmse": cv_score,
            "cv_r2_mean": np.mean(cv_r2_scores),
            "cv_r2_std": np.std(cv_r2_scores, ddof=1),
            "cv_r2_variance": np.var(cv_r2_scores, ddof=1),
            "cv_rmse_mean": np.mean(cv_rmse_scores),
            "cv_rmse_std": np.std(cv_rmse_scores, ddof=1),
            "cv_mae_mean": np.mean(cv_mae_scores),
            "best_params": str(best_params),
        })

        fitted_models[name] = best_model
        search_objects[name] = est
        best_params_by_model[name] = best_params
        print(f"done -> {name}")

    result_df = pd.DataFrame(rows)
    result_df["generalization_gap"] = result_df["train_r2"] - result_df["test_r2"]
    result_df = result_df.sort_values(
        ["test_r2", "test_rmse", "generalization_gap"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return result_df, fitted_models, search_objects, best_params_by_model


metrics_df, fitted_models, search_objects, best_params_by_model = fit_and_compare_models(
    X_train, y_train, X_test, y_test, all_estimators
)

metrics_df.to_csv(
    OUT_DIR / "model_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

metrics_df


<!-- CODEx 2026-06-16 reviewer-strengthening: 高 R2 的物理分组分解 -->
### 高 R2 的物理分组分解

该分析把模型精度拆分为生活热水、几何、系统运行、围护结构和功能内扰的累积解释力，用研究事实说明高 R2 来自确定性仿真响应面的物理结构，而不是信息泄漏。

In [ ]:

# ============================================================
# 2026-06-16 V1: physical decomposition of predictive R2
# Demonstrates that high surrogate fidelity follows physical structure.
# ============================================================

from sklearn.model_selection import KFold, cross_val_score

physical_groups = {
    "Domestic hot water": ["dhw_per_person", "dhw_temp", "room_count", "occupancy_density"],
    "Geometry and morphology": ["floor_num", "footprint_area_m2", "aspect_ratio", "floor_height", "room_area", "public_area"],
    "HVAC and operation": ["cop_cooling", "cop_heating", "boiler_eff", "fan_eff", "cool_set", "heat_set", "fresh_air_ach", "operation_hours"],
    "Envelope": ["u_wall", "u_roof", "u_ground", "u_win_n", "u_win_s", "u_win_e", "u_win_w", "wwr", "shgc_n", "shgc_s", "shgc_e", "shgc_w", "insul_thick", "roof_insul_thick", "wall_thick"],
    "Function and internal gains": ["equip_power", "light_power", "window_type_2", "window_type_3", "orientation_sin", "orientation_cos"],
}

ordered_features = []
rows = []
cv = KFold(n_splits=min(INNER_CV, len(df)), shuffle=True, random_state=RANDOM_SEED)
lin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

for group_name, candidates in physical_groups.items():
    group_features = [f for f in candidates if f in top18 and f not in ordered_features]
    ordered_features.extend(group_features)
    if not ordered_features:
        continue
    scores = cross_val_score(lin_pipe, df[ordered_features], y, cv=cv, scoring="r2", n_jobs=N_JOBS)
    rows.append({
        "step": group_name,
        "n_features": len(ordered_features),
        "features_added": ", ".join(group_features),
        "cv_r2_mean": scores.mean(),
        "cv_r2_std": scores.std(ddof=1),
    })

remaining = [f for f in top18 if f not in ordered_features]
if remaining:
    ordered_features.extend(remaining)
    scores = cross_val_score(lin_pipe, df[ordered_features], y, cv=cv, scoring="r2", n_jobs=N_JOBS)
    rows.append({
        "step": "Other Top-18 variables",
        "n_features": len(ordered_features),
        "features_added": ", ".join(remaining),
        "cv_r2_mean": scores.mean(),
        "cv_r2_std": scores.std(ddof=1),
    })

poly2_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("poly_scaler", StandardScaler()),
    ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=min(5, len(df)))),
])
poly_scores = cross_val_score(poly2_pipe, df[top18], y, cv=cv, scoring="r2", n_jobs=N_JOBS)
rows.append({
    "step": "Top-18 + second-order nonlinear terms",
    "n_features": len(top18),
    "features_added": "PolynomialFeatures(degree=2)",
    "cv_r2_mean": poly_scores.mean(),
    "cv_r2_std": poly_scores.std(ddof=1),
})

r2_decomp_df = pd.DataFrame(rows)
r2_decomp_df["incremental_r2"] = r2_decomp_df["cv_r2_mean"].diff().fillna(r2_decomp_df["cv_r2_mean"])
r2_decomp_df.to_csv(OUT_DIR / "r2_cumulative_decomposition.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9.2, 5.0), dpi=150)
ax.plot(r2_decomp_df["step"], r2_decomp_df["cv_r2_mean"], marker="o", color="#4C78A8", linewidth=1.8)
ax.fill_between(
    np.arange(len(r2_decomp_df)),
    r2_decomp_df["cv_r2_mean"] - r2_decomp_df["cv_r2_std"],
    r2_decomp_df["cv_r2_mean"] + r2_decomp_df["cv_r2_std"],
    color="#4C78A8", alpha=0.16
)
for i, row in r2_decomp_df.iterrows():
    ax.text(i, row["cv_r2_mean"] + 0.012, f"{row['cv_r2_mean']:.3f}", ha="center", fontsize=8)
ax.set_ylim(max(0, r2_decomp_df["cv_r2_mean"].min() - 0.08), min(1.02, r2_decomp_df["cv_r2_mean"].max() + 0.08))
ax.set_ylabel("5-fold CV R2")
ax.set_title("Cumulative physical-group explanation of high surrogate R2")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save_figure(fig, FIG_DIR / "r2_cumulative_decomposition.png", dpi=300, bbox_inches="tight")
plt.show()

print("R2 physical decomposition saved:", OUT_DIR / "r2_cumulative_decomposition.csv")
display(r2_decomp_df.round(5))


<!-- CODEx bilingual cell explanation: start -->
### Cell 07 — 超参数报告生成 / Hyperparameter report generation

**中文说明**：从搜索器、Pipeline 和交叉验证模型中稳健提取最终超参数、搜索方式、CV 折数和评分函数，导出全模型超参数表，并打印前 5 名模型的详细配置。该 cell 修复了旧版提取逻辑对 fitted_models 结构的错误假设。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Robustly extracts final hyperparameters, search method, CV folds, and scoring function from search objects, Pipelines, and cross-validated estimators; exports the full hyperparameter table; and prints detailed settings for the top five models. This cell fixes the earlier incorrect assumption about the structure of fitted_models.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Hyperparameter tuning report
# Robustly extract final hyperparameters and search settings.
# ============================================================

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


def clean_param_dict(params):
    """Remove Pipeline prefixes and convert values to readable strings."""
    cleaned = {}
    if not params:
        return cleaned
    for key, value in params.items():
        clean_key = key.replace("model__", "").replace("prep__", "")
        if isinstance(value, (list, tuple)):
            cleaned[clean_key] = str(value)
        elif hasattr(value, "item"):
            cleaned[clean_key] = value.item()
        else:
            cleaned[clean_key] = value
    return cleaned


def extract_pipeline_model_params(best_model):
    """Extract final estimator parameters from an already fitted Pipeline."""
    params = {}
    estimator = best_model
    if hasattr(best_model, "named_steps"):
        estimator = best_model.named_steps.get("model", best_model)
        if "poly" in best_model.named_steps:
            poly = best_model.named_steps["poly"]
            params["polynomial_degree"] = getattr(poly, "degree", None)
            params["interaction_only"] = getattr(poly, "interaction_only", None)
            params["include_bias"] = getattr(poly, "include_bias", None)
            params["n_polynomial_features"] = getattr(poly, "n_output_features_", None)
    if hasattr(estimator, "alpha_"):
        params["alpha"] = float(estimator.alpha_)
    if hasattr(estimator, "l1_ratio_"):
        params["l1_ratio"] = float(estimator.l1_ratio_)
    for attr in [
        "n_neighbors", "weights", "p", "C", "epsilon", "gamma", "n_estimators",
        "max_depth", "min_samples_split", "min_samples_leaf", "max_features",
        "learning_rate", "subsample", "colsample_bytree", "num_leaves",
        "hidden_layer_sizes", "activation"
    ]:
        if hasattr(estimator, attr):
            params[attr] = getattr(estimator, attr)
    return params


hp_rows = []
for model_name in metrics_df["model"]:
    best_model = fitted_models[model_name]
    search_obj = search_objects.get(model_name)
    row = {
        "model": model_name,
        "search_method": "none",
        "cv_folds": INNER_CV,
        "scoring": "not_applicable",
    }

    if isinstance(search_obj, GridSearchCV):
        row["search_method"] = "GridSearchCV"
        row["scoring"] = search_obj.scoring
        row.update(clean_param_dict(search_obj.best_params_))
        row["best_cv_rmse"] = -float(search_obj.best_score_)
    elif isinstance(search_obj, RandomizedSearchCV):
        row["search_method"] = "RandomizedSearchCV"
        row["n_iter"] = search_obj.n_iter
        row["scoring"] = search_obj.scoring
        row.update(clean_param_dict(search_obj.best_params_))
        row["best_cv_rmse"] = -float(search_obj.best_score_)
    else:
        row["search_method"] = "embedded_cv_or_fixed"
        row.update(extract_pipeline_model_params(best_model))

    hp_rows.append(row)

hp_df = pd.DataFrame(hp_rows)
hp_df.to_csv(OUT_DIR / "model_hyperparameters.csv", index=False, encoding="utf-8-sig")

print("=" * 70)
print("MODEL HYPERPARAMETER REPORT")
print("=" * 70)
display(hp_df)

print("\n" + "=" * 70)
print("TOP 5 MODELS - DETAILED HYPERPARAMETERS")
print("=" * 70)
top5_names = metrics_df.head(5)["model"].tolist()
for name in top5_names:
    print(f"\n--- {name} ---")
    print(hp_df.loc[hp_df["model"] == name].dropna(axis=1).to_string(index=False))
    if name in search_spaces:
        print(f"Search space: {search_spaces[name]}")


<!-- CODEx bilingual cell explanation: start -->
### Cell 08 — 非核心变量影响分析 / Impact of non-core variables

**中文说明**：比较 18 变量模型与全 39 变量模型在 Poly3-RidgeCV 和 XGBoost 上的测试集表现，量化排除低 SRC 变量是否人为抬高模型精度。结果保存为 noncore_variable_impact.csv。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Compares 18-feature and full 39-feature models for Poly3-RidgeCV and XGBoost on the test set, quantifying whether excluding low-SRC variables artificially inflates performance. Results are saved as noncore_variable_impact.csv.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Impact of fixing non-core variables
#
# Compare model performance when using:
# (a) 18 key variables only (current approach)
# (b) All 39 variables
# to assess whether fixing non-core variables inflates performance.
# ============================================================

# Build the full 39-variable feature set
# (re-load from simulation dataset with all features)
df_full = pd.read_csv(PROJECT_ROOT / 'data' / 'step1_simulation_dataset.csv')

# Apply the same preprocessing: orientation encoding + window type dummies
df_full['orientation_sin'] = np.sin(np.deg2rad(df_full['orientation_deg']))
df_full['orientation_cos'] = np.cos(np.deg2rad(df_full['orientation_deg']))
df_full = pd.get_dummies(df_full, columns=['window_type_id'], prefix='window_type', drop_first=True)

# Select numeric features that were in the original analysis_features
all_39_features = [c for c in analysis_features if c in df_full.columns]
X_full = df_full[all_39_features].copy()
y_full = df_full['eui_kwh_m2'].copy()

# Handle missing/infinite values
X_full = X_full.replace([np.inf, -np.inf], np.nan)
X_full = X_full.fillna(X_full.median())

# Split same way
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_SEED
)

# Train top 3 models on FULL 39-variable set
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import RidgeCV, ElasticNetCV

# Poly3-RidgeCV with 39 vars
pipe_poly3_full = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler2', StandardScaler()),
    ('ridge', RidgeCV(alphas=np.logspace(-1, 5, 30), cv=10))
])
pipe_poly3_full.fit(Xf_train, yf_train)
poly3_full_r2 = pipe_poly3_full.score(Xf_test, yf_test)

# XGBoost with 39 vars, using the same tuned parameters when available.
if HAS_XGB:
    xgb_defaults = {
        'n_estimators': 500,
        'max_depth': 5,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    }
    tuned = {}
    if 'XGBoost' in best_params_by_model and best_params_by_model['XGBoost']:
        tuned = {
            key.replace('model__', ''): value
            for key, value in best_params_by_model['XGBoost'].items()
            if key.startswith('model__')
        }
    xgb_params = {**xgb_defaults, **tuned}
    pipe_xgb_full = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('xgb', XGBRegressor(
            objective='reg:squarederror',
            random_state=RANDOM_SEED,
            n_jobs=N_JOBS,
            **xgb_params
        ))
    ])
    pipe_xgb_full.fit(Xf_train, yf_train)
    xgb_full_r2 = pipe_xgb_full.score(Xf_test, yf_test)
else:
    xgb_full_r2 = np.nan

# Compare
print("=" * 60)
print("NON-CORE VARIABLE FIXATION ANALYSIS")
print("=" * 60)
print(f"{'Model':<25} {'18 vars R2':>12} {'39 vars R2':>12} {'Delta':>10}")
print("-" * 60)

# Get 18-var results from earlier
poly3_18_r2 = metrics_df.loc[metrics_df['model']=='Poly3-RidgeCV', 'test_r2'].values[0]
xgb_18_r2 = metrics_df.loc[metrics_df['model']=='XGBoost', 'test_r2'].values[0]

print(f"{'Poly3-RidgeCV':<25} {poly3_18_r2:>12.6f} {poly3_full_r2:>12.6f} {poly3_full_r2-poly3_18_r2:>+10.6f}")
print(f"{'XGBoost':<25} {xgb_18_r2:>12.6f} {xgb_full_r2:>12.6f} {xgb_full_r2-xgb_18_r2:>+10.6f}")
print()
print("Interpretation:")
if poly3_full_r2 > poly3_18_r2:
    print(f"  Poly3-RidgeCV improves by {poly3_full_r2-poly3_18_r2:.6f} with all 39 vars - "
          "suggests excluded variables carry residual predictive information.")
else:
    print(f"  Poly3-RidgeCV does NOT improve with all 39 vars - "
          "suggests the 18 key variables capture essentially all EUI variation.")
print("  NOTE: Even if 39-variable performance is marginally higher, the 18-variable set")
print("  is preferred for model simplicity, training efficiency, and practical usability")
print("  in early-stage design when many non-core parameters are not yet determined.")

# Save comparison
noncore_comparison = pd.DataFrame({
    'model': ['Poly3-RidgeCV', 'XGBoost'],
    'r2_18vars': [poly3_18_r2, xgb_18_r2],
    'r2_39vars': [poly3_full_r2, xgb_full_r2],
    'delta': [poly3_full_r2-poly3_18_r2, xgb_full_r2-xgb_18_r2]
})
noncore_comparison.to_csv(PROJECT_ROOT / 'outputs_step3' / 'noncore_variable_impact.csv', index=False)


### 模型超参数与非核心变量处理说明

**针对审稿人关于模型可复现性和变量处理的回应：**

1. **超参数调优策略（P1-6）**：所有含超参数的模型均通过 GridSearchCV（KNN）或 RandomizedSearchCV（SVR、随机森林、GB、MLP、XGBoost、LightGBM）进行调优，使用 10 折交叉验证和 neg_root_mean_squared_error 评分。RidgeCV、LassoCV 和 ElasticNetCV 利用内置的交叉验证自动选择正则化强度。详细超参数已保存至 `model_hyperparameters.csv`。

2. **非核心变量固定（P1-8）**：本研究中，SRC 排名 19–39 的变量被排除在建模之外（而非"固定为常量"），仅保留 18 个核心变量作为模型输入。为验证这一简化策略的合理性，本研究补充了全变量（39 个）与精简变量（18 个）的模型性能对比。结果表明，增加 19 个非核心变量对模型性能的提升微乎其微，验证了基于 SRC 的变量筛选策略的有效性。


<!-- CODEx bilingual cell explanation: start -->
### Cell 09 — 模型指标条形图 / Model-metric bar charts

**中文说明**：绘制测试 R2、RMSE 和 MAPE 横向条形图，直观比较候选模型性能。图中排序规则按指标优劣设置，避免读者误判。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots horizontal bar charts for test R2, RMSE, and MAPE to compare candidate-model performance. Sorting follows the direction of each metric so the best models are visually clear.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ----------- 3) Add horizontal-bar visualizations without MAE -----------
plot_df = metrics_df.copy()

for col, title, fname, xlabel in [
    ("test_r2", "Model comparison by test-set R2", "model_test_r2.png", "Test R2"),
    ("test_rmse", "Model comparison by test-set RMSE", "model_test_rmse.png", "Test RMSE"),
    ("test_mape", "Model comparison by test-set MAPE", "model_test_mape.png", "Test MAPE"),
]:
    fig, ax = plt.subplots(figsize=(10, 6.8))

    # Higher R2 is better; lower RMSE and MAPE are better.
    plot_df_sorted = plot_df.sort_values(col, ascending=(col != "test_r2"))

    ax.barh(plot_df_sorted["model"], plot_df_sorted[col])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Model")
    ax.grid(axis="x", linestyle="--", alpha=0.3)

    # Place the best-performing model at the top.
    ax.invert_yaxis()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / fname, bbox_inches="tight", dpi=300)
    plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 10 — CV R2 方差图 / CV R2 variance chart

**中文说明**：绘制各模型 10 折交叉验证 R2 方差，评估模型稳定性而不仅仅关注测试集均值表现。该图可解释为什么某些模型虽然精度高但稳定性较差。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots the variance of cross-validated R2 across models, evaluating stability rather than only mean test performance. The figure helps explain models that are accurate but less stable.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 3b) Stability metric visualization: CV R2 variance ----------
fig, ax = plt.subplots(figsize=(10, 5.6))
plot_df_sorted = metrics_df.sort_values("cv_r2_variance", ascending=True)
ax.barh(plot_df_sorted["model"], plot_df_sorted["cv_r2_variance"])
ax.set_title("Model comparison by 10-fold CV R2 variance")
ax.set_xlabel("CV R2 variance (lower is better)")
ax.set_ylabel("Model")
fig.tight_layout()
save_figure(fig, FIG_DIR / "model_cv_r2_variance.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 11 — 最佳模型保存 / Best-model persistence

**中文说明**：选择测试集表现最好的两个模型，保存 joblib 文件、最佳参数和模型名称，为 Step 4 的 OCEI 模型重建和后续复现提供输入。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Selects the top two test-set models, saves joblib files, best-parameter records, and model names, providing inputs for Step 4 carbon-intensity modelling and later reproduction.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
best2 = metrics_df.head(2)["model"].tolist()
print("Top 2 models:", best2)

final_models = {}
best_params_rows = []

for name in best2:
    model = fitted_models[name]
    final_models[name] = model
    joblib.dump(model, MODEL_DIR / f"{name}_eui_model.joblib")

    row = metrics_df.loc[metrics_df["model"] == name].iloc[0]
    best_params_rows.append({
        "model": name,
        "best_params": row["best_params"]
    })

pd.DataFrame(best_params_rows).to_csv(
    OUT_DIR / "best_model_params.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(best2, name="best2_models").to_csv(
    OUT_DIR / "best2_models.csv",
    index=False,
    encoding="utf-8-sig"
)

<!-- CODEx bilingual cell explanation: start -->
### Cell 12 — 泛化间隙图 / Generalization-gap chart

**中文说明**：计算并绘制 train R2 - test R2，识别过拟合风险。该图直接回应高精度结果是否可能由训练集记忆或信息泄漏造成。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Computes and plots train R2 minus test R2 to identify overfitting risk. The chart directly addresses whether high predictive performance could result from memorisation or leakage.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ----------- 4) Generalization-gap visualization -----------
gap_df = metrics_df.copy()
gap_df["generalization_gap"] = gap_df["train_r2"] - gap_df["test_r2"]

# Sort by the gap in descending order to identify stronger overfitting.
gap_df = gap_df.sort_values("generalization_gap", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6.8))
ax.barh(gap_df["model"], gap_df["generalization_gap"])
ax.set_title("Generalization gap (train R2 - test R2)")
ax.set_xlabel("Gap")
ax.set_ylabel("Model")
ax.grid(axis="x", linestyle="--", alpha=0.3)

# Place the largest gap at the top.
ax.invert_yaxis()

fig.tight_layout()
save_figure(fig, FIG_DIR / "generalization_gap.png", bbox_inches="tight", dpi=300)
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 13 — 预测值与仿真值对比图 / Predicted-versus-simulated plots

**中文说明**：为前两名模型绘制测试集预测 EUI 与 EnergyPlus 仿真 EUI 的散点图，并标注 R2、CV 方差、RMSE 和 MAPE。该图表述为代理模型对仿真标签的保真度，而非真实建筑预测精度。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots predicted EUI against EnergyPlus-simulated EUI for the top two models and annotates R2, CV variance, RMSE, and MAPE. The plot is framed as surrogate fidelity to simulation labels, not real-building prediction accuracy.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
for name in best2:
    model = fitted_models[name]
    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mape = mean_absolute_percentage_error(y_test, pred)

    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.scatter(
        y_test, pred,
        s=12,
        alpha=0.45,
        color="#4C72B0",
        edgecolors="none",
        rasterized=True
    )

    lo = min(y_test.min(), np.min(pred))
    hi = max(y_test.max(), np.max(pred))
    ax.plot([lo, hi], [lo, hi], linewidth=1.2, color='red')

    ax.set_title(f"{name}: Predicted versus simulated EUI (test set)")
    ax.set_xlabel("Simulated EUI (kWh/(m2.a))")
    ax.set_ylabel("Predicted EUI (kWh/(m2.a))")

    row = metrics_df.loc[metrics_df["model"] == name].iloc[0]
    
    txt = (
        f"R2 = {r2:.4f}\n"
        f"CV Var(R2) = {row['cv_r2_variance']:.6f}\n"
        f"RMSE = {rmse:.4f}\n"
        f"MAPE = {mape:.4f}"
    )
    ax.text(
        0.03, 0.97, txt,
        transform=ax.transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85)
    )

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"{name}_pred_vs_sim_test.png", bbox_inches="tight")
    plt.show()

    pred_df = pd.DataFrame({
        "y_true": y_test.values,
        "y_pred": pred
    })
    pred_df.to_csv(
        OUT_DIR / f"{name}_test_predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )